# 05 — Director outputs (the curated few)

The presentation set, mirroring the northern `07_director_outputs` and the y2y trail
(`analyses/y2y/21_director_outputs`): only the maps that go on slides, each ONE asset function in
`wolverine_director` drawn through `director_plot.wide_map` — the y2y Act 1 **wide layout** (13.33 × 7.5 in
slide: the whole Y2Y frame at left, zoom insets A and B stacked at right, the key under A, the legend
under B; hillshade + Natural Earth basemap; PNG at 300 dpi + PDF). Colours and words come from
`corridors_mapstyle` (the §3a tokens); the frame draws at 600 m (one 300 dpi pixel of the frame panel is
~1.5 km). Existing PAs are the layout's grey layer, the refugia nodes the overlay (filled, outlined,
named in the insets), and the proposed IPCAs a second fill over the corridor land — so corridor land
inside PAs / IPCAs reads as already satisfied by the overlay (W11). Knobs: `wd.WIDE_STYLE` (then
`dp.STYLE`), `wd.STYLE`; `INSETS` = "examples" (A = the northernmost example link, B = the southernmost)
or "clusters" (the two densest node clusters).

The full record (every asset, tables, GIS, QA) is `04_tables_and_figures.ipynb`.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

import matplotlib.pyplot as plt
import corridors_mapstyle as ms          # the §3a tokens: class + cost colours and words
import corridors_director as cd
import director_plot as dp               # the y2y Act 1 wide layout (one codebase with analyses/y2y/21 and the northern 07)
import wolverine_director as wd
for _m in (ms, cd, dp, wd):
    importlib.reload(_m)

RUN = "v2_run001"                        # <-- the run to present
INSETS = "examples"                      # "examples" | "clusters"
R = cc.load_results(config.RESULTS_DIR / CFG["results_subdir"] / RUN)
P = wd.package(R)
F = wd.director_frame(P)                 # the run as a director_plot frame: 600 m grid, PAs grey, the refugia nodes as the overlay, the Y2Y window
dp.STYLE.update(wd.WIDE_STYLE)           # the wolverine knobs for the wide layout -- tweak dp.STYLE HERE, after this line, before drawing
plt.rcParams.update(dp.SPEC_RC)
OUT = P.out / "director_outputs"; OUT.mkdir(exist_ok=True)
print(f"frame {F.G.shape} @ {1000 / F.PX_PER_KM:.0f} m · window px {tuple(round(v) for v in F.WINDOW)} · insets {INSETS} → {OUT}")

## 01 — the movement-cost surface

What the land is made of on the surface the corridors were routed on ("O'Brien 2025, generic terrain rules
withheld"): the four movement-cost classes in the M0b magma swatches (1 intact · 10 roads and cuts · 100
converted · 1000 water, settlement and the retained barriers), existing PAs grey, proposed IPCAs filled, the
refugia nodes filled + outlined, postal codes on the frame, names + towns in the insets. The class shares
print for the slide's subtitle.

In [ ]:
wd.figure_cost_wide(P, OUT / "01_movement_cost.png", insets=INSETS)

## 02 — where the land still offers choices

The corridor network by routing class = **corridor pressure** (Minimum (options) · Some (narrowing) · A lot
(last affordable) · Maximum (only viable connection); the §3a palette and words), the key in the ramp slot
under inset A; the same frame, insets and legend as 01. PA and IPCA fills lie over the corridor land, so
what is already satisfied within protected land shows as such; the counts of links already connected
within existing PAs / once the proposed IPCAs are realized print for the slide. While H8 is open the
squeezed class is folded into Minimum, exactly as on the record's W1.

In [ ]:
wd.figure_choices_wide(P, OUT / "02_where_choices.png", insets=INSETS)

## 03 — the route options

The deck's numbered examples (up to four, the automatic rule: the link under the most pressure in each act, then a
"room to choose" link; or your pins in `wd.EXAMPLE_PICKS`), numbered north → south. Each option = one link's full
corridor band, filled in its number's colour from the y2y cluster palette over the pressure classes, numbered at the
band's median cell; the PA / IPCA / node fills sit on top, as on the northern 07 · 03. Inset A holds options 1–2 and B
holds 3–4 (one each when a pair does not fit one window). The legend entry is the y2y cluster swatch, "Route options".

In [ ]:
wd.figure_options_wide(P, OUT / "03_route_options.png", insets=INSETS)

## 04 — the route options: star plots + locators

The y2y Act 1 stars (`director_plot.star_grid`, one star per option in its colour): each axis = the option's mean
PERCENTILE of a theme against the Y2Y-wide allocatable landscape (`director_core.block_percentiles`; dashed ring 0.5 =
the typical unprotected cell; fractional 300 m → 1 km cover). Under them, the two inset windows A and B as locator
panels on the star grid's geometry. The first call computes the y2y layers' percentiles (~1–2 min) and caches them on `P`.

In [ ]:
wd.option_stars(P, OUT / "04_route_option_stars.png")
wd.option_locators(P, OUT / "04b_route_option_locators.png", insets=INSETS, panel_px=620)   # + each panel as its own file (_A, _B)

## 05 — the route options: consequences

The y2y consequences table (`director_plot.consequences_table`, transposed, per-row red → blue fills over the option
columns): mean raw value in the option's land ÷ mean over Y2Y-wide allocatable land, per theme ("2.3×" = 2.3 times the
average unprotected cell), columns = the options then two reference areas — Banff National Park (existing PA) and
Dene Kʼéh Kusān (proposed IPCA), the y2y tables' rule (`wd.CONSEQ_REFERENCE`). Rows also to
`director_package/tables/route_option_consequences.csv`.

In [ ]:
wd.option_consequences(P, OUT / "05_route_option_consequences.png")